# Breast Cancer Diagnosis — Machine Learning Classification

**Objective:** Predict whether a breast mass is *malignant* or *benign* using the Wisconsin Diagnostic Breast Cancer dataset.

> Educational demonstration only. This model is not a medical diagnostic tool and must not be used for clinical decisions.

## Workflow
1. Load and inspect the dataset
2. Clean data and encode the target
3. Explore class balance and detect potential outliers
4. Split data into training and test sets
5. Build preprocessing pipelines (imputation, scaling, feature selection)
6. Train and compare three classification algorithms
7. Evaluate using accuracy, precision, recall, F1 and ROC-AUC
8. Visualize results

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, roc_auc_score
)

## 1. Load the dataset
The CSV file is included in this ZIP. Keep it in the same folder as this notebook.

In [ ]:
df = pd.read_csv("breast_cancer_dataset.csv")
print("Dataset shape:", df.shape)
display(df.head())
print("\\nMissing values per column:")
display(df.isnull().sum().sort_values(ascending=False).head(10))
print("\\nTarget distribution:")
display(df["diagnosis"].value_counts())

## 2. Prepare features and target

The target is encoded as **0 = benign** and **1 = malignant**. The feature columns are numeric measurements. The original dataset has no missing values, but median imputation is included in each pipeline so the workflow remains robust if missing values are introduced later.

In [ ]:
X = df.drop(columns=["diagnosis"])
y = (df["diagnosis"] == "malignant").astype(int)

print("Features:", X.shape[1])
print("Malignant cases (1):", int(y.sum()))
print("Benign cases (0):", int((y == 0).sum()))

## 3. Exploratory analysis and outlier inspection

The boxplot uses the first five numeric features as an example. Outliers are inspected visually rather than automatically removed: extreme measurements can contain useful clinical signal, so deleting them without domain justification could harm the model.

In [ ]:
plt.figure(figsize=(10, 5))
sns.boxplot(data=X.iloc[:, :5])
plt.title("Outlier inspection for first five features")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

plt.figure(figsize=(5, 4))
sns.countplot(x=df["diagnosis"])
plt.title("Class distribution")
plt.xlabel("Diagnosis")
plt.tight_layout()
plt.show()

## 4. Train-test split

A stratified 80:20 split preserves the approximate class proportions in both sets. `random_state=42` makes the experiment reproducible.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

## 5. Preprocessing and model training

Each model uses a pipeline to prevent data leakage: imputation, scaling where appropriate, and feature selection are fitted using training data only. Logistic Regression uses standardization because it is scale-sensitive. Tree-based models do not require feature scaling, so their pipelines omit it. SelectKBest keeps the 15 highest-scoring features.

In [ ]:
models = {
    "Logistic Regression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("selector", SelectKBest(score_func=f_classif, k=15)),
        ("model", LogisticRegression(max_iter=3000, random_state=42))
    ]),
    "Random Forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("selector", SelectKBest(score_func=f_classif, k=15)),
        ("model", RandomForestClassifier(
            n_estimators=300, random_state=42, class_weight="balanced"
        ))
    ]),
    "Gradient Boosting": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("selector", SelectKBest(score_func=f_classif, k=15)),
        ("model", GradientBoostingClassifier(random_state=42))
    ])
}

results = {}
for name, pipeline in models.items():
    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)
    probabilities = pipeline.predict_proba(X_test)[:, 1]

    results[name] = {
        "pipeline": pipeline,
        "predictions": predictions,
        "probabilities": probabilities,
        "accuracy": accuracy_score(y_test, predictions),
        "precision": precision_score(y_test, predictions, zero_division=0),
        "recall": recall_score(y_test, predictions, zero_division=0),
        "f1": f1_score(y_test, predictions, zero_division=0),
        "roc_auc": roc_auc_score(y_test, probabilities)
    }

print("Training completed for", len(models), "models.")

## 6. Compare model performance

In [ ]:
comparison = pd.DataFrame([
    {
        "Model": name,
        "Accuracy": values["accuracy"],
        "Precision": values["precision"],
        "Recall": values["recall"],
        "F1 Score": values["f1"],
        "ROC-AUC": values["roc_auc"]
    }
    for name, values in results.items()
]).sort_values("F1 Score", ascending=False)

display(comparison.round(4))

## 7. Confusion matrix and classification report

Recall is particularly important in this educational example because a false negative means a malignant case was predicted as benign. Always interpret metrics in context.

In [ ]:
best_model_name = comparison.iloc[0]["Model"]
best_predictions = results[best_model_name]["predictions"]

print("Best model by F1 score:", best_model_name)
print(classification_report(
    y_test, best_predictions,
    target_names=["Benign", "Malignant"],
    zero_division=0
))

cm = confusion_matrix(y_test, best_predictions)
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=["Benign", "Malignant"],
    yticklabels=["Benign", "Malignant"]
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title(f"Confusion Matrix — {best_model_name}")
plt.tight_layout()
plt.show()

## 8. ROC curves

In [ ]:
plt.figure(figsize=(7, 5))
for name, values in results.items():
    fpr, tpr, thresholds = roc_curve(y_test, values["probabilities"])
    plt.plot(fpr, tpr, label=f"{name} (AUC={values['roc_auc']:.3f})")

plt.plot([0, 1], [0, 1], "k--", label="Random baseline")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison")
plt.legend()
plt.tight_layout()
plt.show()

## 9. Conclusions

- Three classification algorithms were trained and evaluated on the same stratified test set.
- Compare precision, recall and F1 score rather than relying on accuracy alone.
- Feature selection was performed inside each pipeline to avoid information leakage.
- Outliers were inspected but not blindly removed because extreme values may be meaningful.
- This is a learning exercise, not a clinical diagnostic system.